# Happy LLM — обучение с нуля на Kaggle

Отдельный эксперимент: маленькая **своя** языковая модель на русских диалогах.
Не трогает основного бота. Всё обучение — в `llm/train.py`.

**Ничего не теряется:**
1. чекпоинт каждые 500 шагов в `/kaggle/working/happy/llm/checkpoints/`;
2. каждые 1000 шагов — пуш в HF `{run}/ckpt/latest.pt`;
3. `--resume` продолжает с последнего шага;
4. в конце — `llm_model.pt`, `tokenizer.json`, `metrics.json`, `samples.txt`, `DONE`.

Секреты: `HF_TOKEN`, `HF_REPO_LLM=vaprooll/Happy-LLM`, `GH_REPO=qtttyr/Happy`.


In [ ]:
!pip install -q tokenizers


In [ ]:
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient

SMOKE = True   # первый раз True (3–5 минут), потом False

s = UserSecretsClient()
HF_TOKEN = s.get_secret("HF_TOKEN")
HF_REPO = s.get_secret("HF_REPO_LLM")  # отдельный репозиторий для LLM
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_REPO"] = HF_REPO

try:
    GH_REPO = s.get_secret("GH_REPO")
except Exception:
    GH_REPO = "qtttyr/Happy"

work = "/kaggle/working/happy"
if not os.path.exists(work):
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{GH_REPO}.git", work], check=True)
os.chdir(work)
print("repo:", GH_REPO, "| hf:", HF_REPO)


In [ ]:
if SMOKE:
    data_cmd = [
        sys.executable, "llm/prepare_data.py",
        "--source", "local", "--max-pairs", "3000", "--smoke",
        "--out", "llm/data_smoke",
    ]
else:
    data_cmd = [
        sys.executable, "llm/prepare_data.py",
        "--source", "hf", "--max-pairs", "1000000",
        "--hf-repo", HF_REPO, "--pull", "--push",
    ]
print(" ".join(data_cmd))
subprocess.run(data_cmd, check=True)


In [ ]:
if SMOKE:
    train_cmd = [
        sys.executable, "llm/train.py",
        "--smoke", "--data-dir", "llm/data_smoke",
        "--run-name", "smoke", "--hf-repo", HF_REPO, "--resume",
    ]
else:
    train_cmd = [
        sys.executable, "llm/train.py",
        "--data-dir", "llm/data",
        "--run-name", "llm-main",
        "--hf-repo", HF_REPO, "--resume", "--max-hours", "8",
    ]
print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True)


In [ ]:
from pathlib import Path
run = "smoke" if SMOKE else "llm-main"
print(Path(f"llm/artifacts/{run}/metrics.json").read_text())
print(Path(f"llm/artifacts/{run}/samples.txt").read_text())


## Если сессия упала

1. Открой этот же ноутбук.
2. Снова **Save & Run All (Commit)**.
3. `--resume` подтянет `{run}/ckpt/latest.pt` из HF.
4. `run-name` не меняй: `llm-main` для полного, `smoke` для smoke.

## Важно

- Не меняй `--max-hours` выше 8: нужно время на финальное сохранение.
- Не удаляй `llm/checkpoints/` внутри `/kaggle/working`.
- Если HF push выключен, в логах будет `hf push: OFF`.
